# LLM Token Compression Fine-Tuning
Fine-tune Qwen2.5-0.5B to decompress text compressed with reference tokens using LoRA.

**Setup:** Upload `training_pairs.json` to `/kaggle/input/token-compression-data/`

In [1]:
# Cell 1: Install Dependencies
!pip install -q transformers accelerate peft trl datasets torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 30.9 MB/s eta 0:00:00


In [2]:
# Cell 2: Imports & Device Setup
import json
import torch
import sys
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple
from datasets import Dataset
from peft import LoraConfig as PeftLoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- CRITICAL GPU CHECK ---
if not torch.cuda.is_available():
    print("\n" + "!" * 50)
    print("ERROR: GPU NOT DETECTED!")
    print("Please enable GPU in Kaggle: Sidebar -> Notebook Options -> Accelerator -> GPU T4 x2")
    print("!" * 50 + "\n")
    # Stop execution here if no GPU to prevent slow CPU training
    sys.exit("Execution stopped: No GPU available.")

DEVICE = "cuda"
print(f"Using device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

MODEL_NAME = "Qwen/Qwen2.5-1.5B"
DATA_PATH = "/content/training_pairs.json"
OUTPUT_DIR = "./lora_adapter"

Using device: cuda
GPU: Tesla T4


In [3]:
# Cell 3: token_compression/config.py


@dataclass
class CompressionConfig:
    """Configuration for compression algorithm."""

    window_size: int = 8192
    min_match_length: int = 4
    max_match_length: int = 256
    min_savings: int = 2

In [4]:
# Cell 4: token_compression/dictionary.py

DICTIONARY = {
    # --- Top Frequency 3-grams & 2-grams (Huge Savings) ---
    " of the ": "①",
    " in the ": "②",
    " to the ": "③",
    " on the ": "④",
    " and the ": "⑤",
    " for the ": "⑥",
    " at the ": "⑦",
    " is the ": "⑧",
    " with the ": "⑨",
    " from the ": "⑩",
    " by the ": "⑪",
    " as the ": "⑫",
    # --- Structural/Common Words ---
    "the ": "⑬",
    "and ": "⑭",
    "that ": "⑮",
    "with ": "⑯",
    "from ": "⑰",
    "have ": "⑱",
    "this ": "⑲",
    "will ": "⑳",
    "your ": "Ⓐ",
    "they ": "Ⓑ",
    "been ": "Ⓒ",
    "were ": "Ⓓ",
    "would ": "Ⓔ",
    "could ": "Ⓕ",
    "should ": "Ⓖ",
    "there ": "Ⓗ",
    "their ": "Ⓘ",
    "which ": "Ⓙ",
    "about ": "Ⓚ",
    "other ": "Ⓛ",
    "these ": "Ⓜ",
    "those ": "Ⓝ",
    "being ": "Ⓞ",
    "through ": "Ⓟ",
    "because ": "Ⓠ",
    "between ": "Ⓡ",
    "however ": "Ⓢ",
    "without ": "Ⓣ",
    "another ": "Ⓤ",
    "something ": "Ⓥ",
    "different ": "Ⓦ",
    "important ": "Ⓧ",
    "while ": "Ⓨ",
    "where ": "Ⓩ",
    "every ": "ⓐ",
    "start ": "ⓑ",
    "after ": "ⓒ",
    "before ": "ⓓ",
    "thing ": "ⓔ",
    "place ": "ⓕ",
    "great ": "ⓖ",
    "right ": "ⓗ",
    "again ": "ⓘ",
    "small ": "ⓙ",
    "large ": "ⓚ",
    "found ": "ⓛ",
    "sound ": "ⓜ",
    "under ": "ⓝ",
    "still ": "ⓞ",
    "never ": "ⓟ",
    "house ": "ⓠ",
    "world ": "ⓡ",
    "below ": "ⓢ",
    "asked ": "ⓣ",
    "going ": "ⓤ",
    "large ": "ⓥ",
    "until ": "ⓦ",
    "along ": "ⓧ",
    "shall ": "ⓨ",
    "being ": "ⓩ",
    "often ": "0",
    "whole ": "1",
    "among ": "2",
}

REVERSE_DICTIONARY = {v: k for k, v in DICTIONARY.items()}
BASE62 = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"


def to_base62(n: int) -> str:
    if n == 0:
        return "0"
    result = ""
    while n:
        result = BASE62[n % 62] + result
        n //= 62
    return result


def from_base62(s: str) -> int:
    result = 0
    for c in s:
        result = result * 62 + BASE62.index(c)
    return result


print(f"Dictionary expanded to: {len(DICTIONARY)} phrases")

Dictionary expanded to: 73 phrases


In [5]:
# Cell 5: token_compression/compressor.py


class Compressor:
    """Text compression logic optimized for LLM processing."""

    def __init__(self, config: Optional[CompressionConfig] = None):
        self.config = config or CompressionConfig()
        self._reset_stats()

    def _reset_stats(self):
        self.stats = {"original_chars": 0, "compressed_chars": 0, "dict_subs": 0, "lz77_refs": 0}

    @staticmethod
    def _make_ref(distance: int, length: int) -> str:
        return f"‹{to_base62(distance)}:{to_base62(length)}›"

    def _apply_dictionary(self, text: str) -> str:
        result = text
        for phrase, token in sorted(DICTIONARY.items(), key=lambda x: -len(x[0])):
            count = result.count(phrase)
            if count > 0:
                result = result.replace(phrase, token)
                self.stats["dict_subs"] += count
        return result

    def _find_best_match(self, text: str, pos: int) -> Optional[Tuple[int, int]]:
        window_start = max(0, pos - self.config.window_size)
        window = text[window_start:pos]
        lookahead = text[pos : pos + self.config.max_match_length]
        if len(lookahead) < self.config.min_match_length:
            return None
        for length in range(min(len(lookahead), self.config.max_match_length), self.config.min_match_length - 1, -1):
            pattern = lookahead[:length]
            match_pos = window.rfind(pattern)
            if match_pos >= 0:
                distance = len(window) - match_pos
                ref = self._make_ref(distance, length)
                if length - len(ref) >= self.config.min_savings:
                    return distance, length
        return None

    def compress(self, text: str) -> str:
        self._reset_stats()
        self.stats["original_chars"] = len(text)
        working = self._apply_dictionary(text)
        output, pos = [], 0
        while pos < len(working):
            match = self._find_best_match(working, pos)
            if match:
                distance, length = match
                output.append(self._make_ref(distance, length))
                pos += length
                self.stats["lz77_refs"] += 1
            else:
                output.append(working[pos])
                pos += 1
        result = "".join(output)
        self.stats["compressed_chars"] = len(result)
        return result

    def get_stats(self) -> Dict:
        orig = max(1, self.stats["original_chars"])
        comp = max(1, self.stats["compressed_chars"])
        return {**self.stats, "compression_ratio": round(orig / comp, 2)}


print("Compressor loaded")

Compressor loaded


In [6]:
# Cell 6: token_compression/decompressor.py


class Decompressor:
    """Decompression logic to expand compressed text."""

    @staticmethod
    def _parse_ref(ref: str) -> Tuple[int, int]:
        inner = ref[1:-1]
        d_str, l_str = inner.split(":")
        return from_base62(d_str), from_base62(l_str)

    @staticmethod
    def _reverse_dictionary(text: str) -> str:
        result = text
        for token, phrase in REVERSE_DICTIONARY.items():
            result = result.replace(token, phrase)
        return result

    @staticmethod
    def _expand_overlap(current: str, distance: int, length: int) -> str:
        start = len(current) - distance
        if start < 0:
            return ""
        copied = ""
        for j in range(length):
            src_idx = start + j
            if src_idx < len(current):
                copied += current[src_idx]
            else:
                copied += copied[j - distance]
        return copied

    def decompress(self, compressed: str) -> str:
        output, i = [], 0
        while i < len(compressed):
            if compressed[i] == "‹":
                end = compressed.find("›", i)
                if end > i:
                    ref = compressed[i : end + 1]
                    try:
                        distance, length = self._parse_ref(ref)
                        current = "".join(output)
                        copied = self._expand_overlap(current, distance, length)
                        if copied:
                            output.append(copied)
                            i = end + 1
                            continue
                    except ValueError:
                        pass
            output.append(compressed[i])
            i += 1
        return self._reverse_dictionary("".join(output))


print("Decompressor loaded")

Decompressor loaded


In [7]:
# Cell 7: token_compression/service.py


class CompressionService:
    """Facade for compression and decompression services."""

    def __init__(self, config: Optional[CompressionConfig] = None):
        self.config = config or CompressionConfig()
        self.compressor = Compressor(self.config)
        self.decompressor = Decompressor()

    def compress(self, text: str) -> str:
        return self.compressor.compress(text)

    def decompress(self, compressed_text: str) -> str:
        return self.decompressor.decompress(compressed_text)

    def get_stats(self) -> Dict:
        return self.compressor.get_stats()


# Test compression
service = CompressionService()
test_text = "The quick brown fox jumps over the lazy dog. The quick brown fox jumps again."
compressed = service.compress(test_text)
decompressed = service.decompress(compressed)
print(f"Original: {test_text}")
print(f"Compressed: {compressed}")
print(f"Stats: {service.get_stats()}")

Original: The quick brown fox jumps over the lazy dog. The quick brown fox jumps again.
Compressed: The quick brown fox jumps over ⑬lazy dog. ‹g:Q›again.
Stats: {'original_chars': 77, 'compressed_chars': 53, 'dict_subs': 1, 'lz77_refs': 1, 'compression_ratio': 1.45}


In [8]:
# Cell 8: fine_tuning/config.py


@dataclass
class LoraConfigParams:
    r: int = 16
    lora_alpha: int = 32
    # CRITICAL: Keep these MLP targets. They are vital for the 1.5B model's logic.
    target_modules: List[str] = field(
        default_factory=lambda: ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    )
    lora_dropout: float = 0.05
    bias: str = "none"


@dataclass
class TrainingConfig:
    # T4 GPU Constraints
    batch_size: int = 2

    # OPTIMIZED FOR 10,000 EXAMPLES
    num_epochs: int = 3  # Reduced from 10. (10k * 3 = 30k samples seen, which is plenty)
    learning_rate: float = 1e-4
    warmup_steps: int = 100  # Increased for stability with more data
    gradient_accumulation_steps: int = 16  # Increased for speed. Effective batch size = 32.

    # Logging adjustments so console isn't flooded
    logging_steps: int = 25
    save_steps: int = 200  # Save checkpoints roughly 3 times per epoch


@dataclass
class FineTuningConfig:
    model_name: str = "Qwen/Qwen2.5-1.5B"
    data_path: Path = field(default_factory=lambda: Path(DATA_PATH))
    output_dir: Path = field(default_factory=lambda: Path(OUTPUT_DIR))
    lora: LoraConfigParams = field(default_factory=LoraConfigParams)
    training: TrainingConfig = field(default_factory=TrainingConfig)


print("Config optimized for 10,000 examples")

Config optimized for 10,000 examples


In [9]:
# Cell 9

PROMPT_TEMPLATE = """Decompress the following text by expanding all reference tokens:

Compressed: {compressed}

Original:"""


def load_training_pairs(data_path: Path) -> List[Dict]:
    with open(data_path, "r", encoding="utf-8") as f:
        return json.load(f)


def format_for_training(pairs: List[Dict]) -> List[Dict]:
    formatted = []
    for pair in pairs:
        text = f"{PROMPT_TEMPLATE.format(compressed=pair['compressed'])} {pair['original']}"
        formatted.append({"text": text})
    return formatted


def create_dataset(data_path: Path, test_split_percentage: float = 0.1) -> Tuple[Dataset, Dataset]:
    """
    test_split_percentage: Float between 0.0 and 1.0 (e.g., 0.15 = 15% validation)
    """
    pairs = load_training_pairs(data_path)
    print(f"Loaded {len(pairs)} training pairs")

    formatted = format_for_training(pairs)
    full_dataset = Dataset.from_list(formatted)

    split = full_dataset.train_test_split(test_size=test_split_percentage, seed=42)

    print(f"Train: {len(split['train'])} examples")
    print(f"Val:   {len(split['test'])} examples ({int(test_split_percentage * 100)}%)")

    return split["train"], split["test"]


print("Dataset module loaded")

Dataset module loaded


In [10]:
# Cell 10: fine_tuning/trainer.py


class DecompressionTrainer:
    """Trainer for fine-tuning LLM on compressed text decompression."""

    def __init__(self, config: FineTuningConfig):
        self.config = config
        self.model = None
        self.tokenizer = None

    def load_model(self):
        print(f"Loading model: {self.config.model_name}")
        print(f"Target device: {DEVICE}")

        self.tokenizer = AutoTokenizer.from_pretrained(self.config.model_name, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # Use device_map='auto' to automatically handle GPU placement
        self.model = AutoModelForCausalLM.from_pretrained(
            self.config.model_name,
            trust_remote_code=True,
            torch_dtype=torch.float16,  # Force float16 for GPU training
            device_map="auto",  # IMPORTANT: This puts layers on GPU
            attn_implementation="eager",
        )
        self._add_special_tokens()
        self._apply_lora()
        print("Model loaded with LoRA adapters")

    def _add_special_tokens(self):
        # DYNAMIC LOADING: Automatically grabs all symbols from your dictionary in Cell 4
        # plus the LZ77 specific brackets.
        lz77_tokens = ["‹", "›"]
        dict_tokens = list(DICTIONARY.values())  # Pulls ⓐ, ⓑ, ①, etc. directly

        all_new_tokens = lz77_tokens + dict_tokens

        # Filter to only ones not already in the tokenizer (safety check)
        tokens_to_add = [
            t for t in all_new_tokens if self.tokenizer.convert_tokens_to_ids(t) == self.tokenizer.unk_token_id
        ]

        if tokens_to_add:
            self.tokenizer.add_tokens(tokens_to_add)
            self.model.resize_token_embeddings(len(self.tokenizer))
            print(f"Smart-Added {len(tokens_to_add)} special tokens from Dictionary")

    def _apply_lora(self):
        lora_cfg = self.config.lora
        peft_config = PeftLoraConfig(
            r=lora_cfg.r,
            lora_alpha=lora_cfg.lora_alpha,
            target_modules=lora_cfg.target_modules,
            lora_dropout=lora_cfg.lora_dropout,
            bias=lora_cfg.bias,
            task_type=TaskType.CAUSAL_LM,
        )
        self.model = get_peft_model(self.model, peft_config)
        self.model.print_trainable_parameters()

    def train(self, train_dataset: Dataset, eval_dataset: Dataset):
        if self.model is None:
            self.load_model()

        tc = self.config.training

        sft_config = SFTConfig(
            output_dir=str(self.config.output_dir),
            num_train_epochs=tc.num_epochs,
            per_device_train_batch_size=tc.batch_size,
            gradient_accumulation_steps=tc.gradient_accumulation_steps,
            learning_rate=tc.learning_rate,
            warmup_steps=tc.warmup_steps,
            fp16=True,  # Always True for GPU training
            bf16=False,
            logging_steps=tc.logging_steps,
            save_steps=tc.save_steps,
            eval_strategy="steps",
            eval_steps=tc.save_steps,
            save_total_limit=2,
            dataset_text_field="text",
        )

        trainer = SFTTrainer(
            model=self.model,
            args=sft_config,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            processing_class=self.tokenizer,
        )

        print("Starting training...")
        trainer.train()
        print("Training complete!")

        self.model.save_pretrained(self.config.output_dir)
        self.tokenizer.save_pretrained(self.config.output_dir)
        print(f"Saved adapter to {self.config.output_dir}")


print("DecompressionTrainer loaded")

DecompressionTrainer loaded


In [ ]:
# Cell 11: Load Data & Train

config = FineTuningConfig(
    data_path=Path(DATA_PATH),
    output_dir=Path(OUTPUT_DIR),
    training=TrainingConfig(
        num_epochs=3,
        batch_size=2,
        gradient_accumulation_steps=16,
    ),
)

train_ds, eval_ds = create_dataset(config.data_path)
print("\nSample training example:")
print(train_ds[0]["text"][:500] + "...")

trainer = DecompressionTrainer(config)
trainer.train(train_ds, eval_ds)

Loaded 10000 training pairs
Train: 9000 examples
Val:   1000 examples (10%)

Sample training example:
Decompress the following text by expanding all reference tokens:

Compressed: Emission is a word ⑮originally comes ⑰Latin. O‹P:A›it means Ⓥ⑮is sent out. Several things can be‹W:9›: The most common use talks Ⓚe‹2F:8›of toxic gases. Those‹D:6› get produced by factories ⑭motors, like ‹D:6› in cars. People talk Ⓚradio ‹1o:8›‹1a:9›are programmes ⑮p‹q:6›can listen to if Ⓑ⑱a radio receiver. There Ⓕbe‹1P:A› of loud noise, someone might be making‹a:B›s‹1A:8›also ar‹1F:G›ight.

Original: Emission is a wor...
Loading model: Qwen/Qwen2.5-1.5B
Target device: cuda


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

... (14 outputs left out)

wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: james544wanjiku (james544wanjiku-aai-labs) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss


In [ ]:
# Cell 12: Evaluate Decompression


def decompress_with_model(model, tokenizer, compressed_text: str, max_new_tokens: int = 256) -> str:
    prompt = PROMPT_TEMPLATE.format(compressed=compressed_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id, do_sample=False
        )
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_output[len(prompt) :].strip()


pairs = load_training_pairs(Path(DATA_PATH))
print("\n" + "=" * 60)
print("EVALUATION")
print("=" * 60)

correct, total = 0, min(10, len(pairs))
for i, pair in enumerate(pairs[:total]):
    predicted = decompress_with_model(trainer.model, trainer.tokenizer, pair["compressed"])
    original = pair["original"]
    is_match = predicted.strip() == original.strip()
    if is_match:
        correct += 1
    print(f"\nSample {i + 1}: {'✓' if is_match else '✗'}")
    print(f"  COMPRESSED: {pair['compressed'][:60]}...")
    print(f"  EXPECTED:   {original[:60]}...")
    print(f"  PREDICTED:  {predicted[:60]}...")

print("\n" + "=" * 60)
print(f"Accuracy: {correct}/{total} ({100 * correct / total:.1f}%)")

In [ ]:
# Cell 13: Download Adapter

!zip -r lora_adapter.zip ./lora_adapter
print("\nAdapter zipped! Download lora_adapter.zip from output.")
!ls -la ./lora_adapter

upload zip to colab